In [ ]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 { color: #4fc3f7 !important; border-bottom: 2px solid #3498db !important; padding-bottom: 4px; }
    b, strong { color: #f48fb1 !important; }
    .highlight { background-color: #2d2d2d !important; padding: 10px; border-left: 4px solid #3498db; margin: 4px 0; color: #d4d4d4; }
    code { background-color: #333 !important; color: #ffcc80 !important; padding: 2px 4px; border-radius: 4px; }
    .dataframe { background-color: #2d2d2d !important; color: #d4d4d4 !important; }
</style>
"""))
print("Environment ready. Dark theme applied.")

In [ ]:
from Get_Go_Emo import get_go
from Get_Isear import get_isr

go_df = get_go()
isear_df = get_isr()

# Map numeric ISEAR labels to emotion names
ISEAR_EMOTION_MAP = {1: "joy", 2: "fear", 3: "anger", 4: "sadness", 5: "disgust", 6: "shame", 7: "guilt"}
isear_df["labels"] = isear_df["labels"].map(ISEAR_EMOTION_MAP)

DATASETS = {"goEmo": go_df, "ISEAR": isear_df}
for name, df in DATASETS.items():
    display(HTML(f"<h2>Dataset: {name}</h2>"))
    display(HTML(f"<div class='highlight'>Shape: {df.shape}</div>"))
    display(df.head(2))

In [ ]:
import unified_hidden_state_probe_v4_2 as probe
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# Monkey-patch evaluate_single to fix per-class metrics
def patched_evaluate_single(y_true, y_pred, classes, probabilities=None, include_per_class=True):
    labels = np.arange(len(classes))
    result = {
        "accuracy": float(probe.accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(probe.balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "mcc": probe.safe_mcc(y_true, y_pred),
        "cohen_kappa": float(probe.cohen_kappa_score(y_true, y_pred, labels=labels)),
        "confusion_matrix": probe.confusion_matrix(y_true, y_pred, labels=labels).tolist(),
        "classification_report": probe.classification_report(y_true, y_pred, labels=labels, target_names=list(classes), output_dict=True, zero_division=0),
    }
    if probabilities is not None:
        try:
            result["log_loss"] = float(probe.log_loss(y_true, probabilities, labels=labels))
        except Exception:
            result["log_loss"] = None
        result["roc_auc_ovr_macro"] = probe.safe_roc_auc_single(y_true, probabilities, len(classes))
        result["average_precision_macro"] = probe.safe_average_precision_single(y_true, probabilities, len(classes))
        ll = result.get("log_loss")
        result["log_loss_score"] = float(np.exp(-min(max(ll, 0.0), 20.0))) if ll is not None else None
        result.update(probe.confidence_metrics(y_true, probabilities, y_pred))
    if include_per_class:
        # Correct per-class calculation using average=None
        precisions = precision_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
        recalls = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
        f1s = f1_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
        result["per_class"] = {
            name: {
                "precision": float(precisions[i]),
                "recall": float(recalls[i]),
                "f1": float(f1s[i]),
                "support": int(np.sum(y_true == i)),
            }
            for i, name in enumerate(classes)
        }
    return result

probe.evaluate_single = patched_evaluate_single

# Monkey-patch evaluate_multi to ensure per-class metrics are numeric
def patched_evaluate_multi(y_true, y_pred, probabilities=None, classes=None):
    result = probe.evaluate_multi_original(y_true, y_pred, probabilities, classes) if hasattr(probe, 'evaluate_multi_original') else probe.evaluate_multi(y_true, y_pred, probabilities, classes)
    if classes is not None and "per_class" in result:
        for cls in result["per_class"]:
            for metric in ["f1", "precision", "recall"]:
                result["per_class"][cls][metric] = float(result["per_class"][cls][metric]) if result["per_class"][cls][metric] is not None else 0.0
    return result

probe.evaluate_multi_original = probe.evaluate_multi
probe.evaluate_multi = patched_evaluate_multi

print("Probe functions patched for robust per-class metrics.")

In [ ]:
GOEMOTIONS_CLASSES = probe.GOEMOTIONS_CLASSES
ISEAR_CLASSES = probe.ISEAR_CLASSES
EXTERNAL_ROOT = Path('/Volumes/Amirali/hidden_states')
EXPERIMENT_ID = 'baseline_v5_001'

goemotions_contract = probe.DatasetContract(
    target_type='goemotions', text_column='auto', label_column='auto',
    id_column='auto', task_type='multi_label', class_order=GOEMOTIONS_CLASSES,
    lenient_provenance=True, require_provenance=False
)
isear_contract = probe.DatasetContract(
    target_type='isear', text_column='auto', label_column='auto',
    id_column='auto', task_type='single_label', class_order=ISEAR_CLASSES,
    lenient_provenance=True, require_provenance=False
)

probes = [
    probe.ProbeSpec(name='linear_logistic', type='logistic', complexity='linear', standardize=True, C=1.0, max_iter=3000, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_1_hidden', type='mlp', complexity='1_hidden', standardize=True, hidden_dims=['0.5d'], learning_rate=1e-3, weight_decay=1e-4, epochs=80, batch_size=256, patience=12, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_2_hidden', type='mlp', complexity='2_hidden', standardize=True, hidden_dims=['0.5d','0.25d'], learning_rate=1e-3, weight_decay=1e-4, epochs=80, batch_size=256, patience=12, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_3_hidden', type='mlp', complexity='3_hidden', standardize=True, hidden_dims=['0.5d','0.25d','0.125d'], learning_rate=1e-3, weight_decay=1e-4, epochs=80, batch_size=256, patience=12, selection_metric='macro_f1'),
]
print('Contracts and probes defined.')

In [ ]:
all_pairs = probe.discover_model_dataset_pairs(EXTERNAL_ROOT, EXPERIMENT_ID)
print(f"Found {len(all_pairs)} model-dataset pairs.")
display(pd.DataFrame(all_pairs))

In [ ]:
dataset_map = {'goEmo': (goemotions_contract, go_df), 'ISEAR': (isear_contract, isear_df)}
entries = []
for pair in all_pairs:
    model, dataset = pair['model'], pair['dataset']
    contract, df = dataset_map.get(dataset, (None, None))
    if contract:
        entries.append({'model': model, 'dataset': dataset, 'contract': contract, 'dataset_df': df})
print(f"Prepared {len(entries)} matrix entries.")

In [ ]:
# =============================================================================
# CELL 7 – LOAD EXISTING MATRIX RESULTS FROM CHECKPOINT (WITH ENCODING FALLBACK)
# =============================================================================
checkpoint_dir = EXTERNAL_ROOT / 'experiments' / EXPERIMENT_ID / 'matrix_checkpoint'
per_entry_dir = checkpoint_dir / 'per_entry_results'

if not per_entry_dir.exists():
    raise FileNotFoundError(f"Checkpoint results directory not found: {per_entry_dir}")

# Helper to read CSV with fallback encodings
def read_csv_robust(file_path: Path) -> pd.DataFrame:
    encodings = ['utf-8', 'latin-1', 'cp1252']
    for enc in encodings:
        try:
            return pd.read_csv(file_path, encoding=enc)
        except UnicodeDecodeError:
            continue
    # Last resort: replace invalid bytes
    return pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')

csv_files = sorted(per_entry_dir.glob('*_layer_probe_results.csv'))
frames = []
for csv_file in csv_files:
    try:
        df = read_csv_robust(csv_file)
    except Exception as e:
        print(f"Failed to read {csv_file.name}: {e}. Skipping.")
        continue

    # Ensure model/dataset columns exist
    if 'model' not in df.columns or 'dataset' not in df.columns:
        # Infer from filename: <model>_<dataset>_layer_probe_results.csv
        stem = csv_file.stem.replace('_layer_probe_results', '')
        parts = stem.split('_')
        # The last part is dataset, rest is model (may contain underscores)
        df['dataset'] = parts[-1]
        df['model'] = '_'.join(parts[:-1])

    # Ensure artifact_dir column exists
    if 'artifact_dir' not in df.columns:
        df['artifact_dir'] = str(csv_file.parent)

    frames.append(df)

if frames:
    full_results = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} result files. Total rows: {len(full_results)}")
    display(full_results.head())
else:
    full_results = pd.DataFrame()
    print("No valid result files could be loaded.")

In [ ]:
if full_results.empty:
    display(HTML("<div class='highlight'>No results loaded. Please run the probe matrix first or check the checkpoint directory.</div>"))
else:
    print(f"Loaded results for {full_results['model'].nunique()} models and {full_results['dataset'].nunique()} datasets.")

In [ ]:
# =============================================================================
# CELL 7 – SUMMARY STATISTICS AND BEST LAYER TABLE
# =============================================================================
if not full_results.empty:
    # Best layer per probe/model/dataset based on test_macro_f1
    best_per_entry = full_results.loc[
        full_results.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()
    ]
    display(HTML("<h2>Best Layer per Probe (Macro-F1)</h2>"))
    display(best_per_entry[["probe", "model", "dataset", "layer_index", "test_macro_f1", "probe_score"]])

    # Pivot table of best Macro-F1
    pivot_best = best_per_entry.pivot_table(
        index=["model", "dataset"], columns="probe", values="test_macro_f1"
    )
    display(HTML("<h2>Best Macro-F1 Matrix</h2>"))
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))
else:
    display(HTML("<div class='highlight'>No results found. Please run the matrix first.</div>"))

In [ ]:
if not full_results.empty:
    best_per_entry = full_results.loc[full_results.groupby(["probe","model","dataset"])["test_macro_f1"].idxmax()]
    display(HTML("<h2>Best Layer per Probe (Macro-F1)</h2>"))
    display(best_per_entry[["probe","model","dataset","layer_index","test_macro_f1","probe_score"]])
    pivot_best = best_per_entry.pivot_table(index=["model","dataset"], columns="probe", values="test_macro_f1")
    display(HTML("<h2>Best Macro-F1 Matrix</h2>"))
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))
else:
    print("No results loaded.")

In [ ]:
# =============================================================================
# CELL 8 – LAYER CURVES FOR A SINGLE MODEL/DATASET
# =============================================================================
if not full_results.empty:
    # Pick the first entry for demonstration
    sample = full_results.iloc[0]
    model, dataset = sample['model'], sample['dataset']
    subset = full_results[(full_results.model == model) & (full_results.dataset == dataset)]

    plt.figure(figsize=(12, 6))
    for probe_name in subset['probe'].unique():
        data = subset[subset['probe'] == probe_name].sort_values('layer_index')
        plt.plot(data['layer_index'], data['test_macro_f1'], marker='o', label=probe_name)
    plt.xlabel('Layer Index')
    plt.ylabel('Test Macro-F1')
    plt.title(f'Layer-wise Macro-F1 for {model} / {dataset}')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
if not full_results.empty:
    sample = full_results.iloc[0]
    model, dataset = sample['model'], sample['dataset']
    subset = full_results[(full_results.model == model) & (full_results.dataset == dataset)]
    plt.figure(figsize=(12,6))
    for probe_name in subset['probe'].unique():
        data = subset[subset['probe'] == probe_name].sort_values('layer_index')
        plt.plot(data['layer_index'], data['test_macro_f1'], marker='o', label=probe_name)
    plt.xlabel('Layer Index'); plt.ylabel('Test Macro-F1')
    plt.title(f'Layer-wise Macro-F1 for {model} / {dataset}')
    plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# =============================================================================
# CELL 9 – HEATMAPS ACROSS ALL MODELS AND PROBES
# =============================================================================
if not full_results.empty:
    # Average macro-F1 per probe and layer across all models/datasets
    pivot = full_results.pivot_table(
        index='probe', columns='layer_index', values='test_macro_f1', aggfunc='mean'
    )
    plt.figure(figsize=(14, 6))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap='viridis', cbar_kws={'label': 'Macro-F1'})
    plt.title('Average Test Macro-F1 Heatmap (all models/datasets)')
    plt.xlabel('Layer Index')
    plt.ylabel('Probe')
    plt.tight_layout()
    plt.show()
else:
    print("No data to plot.")

In [ ]:
if not full_results.empty:
    best_row = full_results.loc[full_results['test_macro_f1'].idxmax()]
    if best_row.get('task_type') == 'single_label' or 'task_type' not in best_row:
        cm_file = Path(best_row['artifact_dir']) / 'models' / best_row['probe'] / f'layer_{int(best_row["layer_index"])}' / 'repeat_0' / 'confusion_matrix_test.npz'
        if cm_file.exists():
            data = np.load(cm_file)
            cm = data['matrix']
            metrics_file = cm_file.parent / 'metrics.json'
            if metrics_file.exists():
                with open(metrics_file) as f:
                    metrics = json.load(f)
                classes = metrics.get('record', {}).get('classes', [str(i) for i in range(cm.shape[0])])
            else:
                classes = [str(i) for i in range(cm.shape[0])]
            plt.figure(figsize=(10,8))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
            plt.title(f'Confusion Matrix – {best_row["model"]}/{best_row["dataset"]} – {best_row["probe"]} @ Layer {int(best_row["layer_index"])}')
            plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.show()
        else:
            print('Confusion matrix file not found.')
    else:
        print('Best entry is multi-label; confusion matrix not available.')

In [ ]:
# =============================================================================
# CELL 10 – CONFUSION MATRIX FOR BEST PROBE-LAYER COMBINATION
# =============================================================================
if not full_results.empty:
    # Find the overall best row by test_macro_f1
    best_row = full_results.loc[full_results['test_macro_f1'].idxmax()]
    model, dataset, probe_name, layer_idx = best_row['model'], best_row['dataset'], best_row['probe'], int(best_row['layer_index'])

    # Locate the saved confusion matrix
    cm_file = Path(best_row['artifact_dir']) / 'models' / probe_name / f'layer_{layer_idx}' / 'repeat_0' / 'confusion_matrix_test.npz'
    if cm_file.exists():
        data = np.load(cm_file)
        cm = data['matrix']
        # Load class names from metrics.json if available
        metrics_file = Path(best_row['artifact_dir']) / 'models' / probe_name / f'layer_{layer_idx}' / 'repeat_0' / 'metrics.json'
        if metrics_file.exists():
            with open(metrics_file) as f:
                metrics = json.load(f)
            classes = metrics.get('record', {}).get('classes', [str(i) for i in range(cm.shape[0])])
        else:
            classes = [str(i) for i in range(cm.shape[0])]

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix – {model}/{dataset} – {probe_name} @ Layer {layer_idx}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        plt.show()
    else:
        print(f'Confusion matrix file not found: {cm_file}')
else:
    print("No data to plot.")

In [ ]:
# =============================================================================
# CELL 11 – SHUFFLED-LABEL CONTROL COMPARISON
# =============================================================================
if not full_results.empty:
    control_frames = []
    for run_dir in full_results['artifact_dir'].unique():
        ctrl_file = Path(run_dir) / 'shuffled_label_controls.csv'
        if ctrl_file.exists():
            ctrl = pd.read_csv(ctrl_file)
            ctrl['model'] = 'unknown'  # will be overwritten if available
            ctrl['dataset'] = 'unknown'
            # We can try to infer from parent path
            # But for simplicity, just append
            control_frames.append(ctrl)

    if control_frames:
        control_df = pd.concat(control_frames, ignore_index=True)
        plt.figure(figsize=(12, 6))
        for probe_name in control_df['probe'].unique():
            sub_ctrl = control_df[control_df['probe'] == probe_name].groupby('layer_index')['control_test_macro_f1'].mean()
            plt.plot(sub_ctrl.index, sub_ctrl.values, linestyle='--', marker='x', label=f'{probe_name} (shuffled)')
        # Overlay true results (averaged)
        for probe_name in full_results['probe'].unique():
            sub_true = full_results[full_results['probe'] == probe_name].groupby('layer_index')['test_macro_f1'].mean()
            plt.plot(sub_true.index, sub_true.values, linestyle='-', marker='o', label=f'{probe_name} (true)')
        plt.xlabel('Layer Index')
        plt.ylabel('Macro-F1')
        plt.title('True vs Shuffled Label Controls')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No control data found.')
else:
    print("No data to plot.")

In [ ]:
if not full_results.empty:
    best_row = full_results.loc[full_results['test_macro_f1'].idxmax()]
    metrics_file = Path(best_row['artifact_dir']) / 'models' / best_row['probe'] / f'layer_{int(best_row["layer_index"])}' / 'repeat_0' / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            metrics = json.load(f)
        per_class = metrics.get('results', {}).get('test', {}).get('per_class', {})
        if per_class:
            df_per_class = pd.DataFrame(per_class).T
            df_metrics = df_per_class[['f1','precision','recall']].apply(pd.to_numeric, errors='coerce')
            plt.figure(figsize=(12,8))
            sns.heatmap(df_metrics, annot=True, fmt=".2f", cmap='viridis')
            plt.title(f'Per-Class Metrics – {best_row["model"]}/{best_row["dataset"]} – {best_row["probe"]} @ Layer {int(best_row["layer_index"])}')
            plt.xlabel('Metric'); plt.ylabel('Emotion Class'); plt.tight_layout(); plt.show()
    else:
        print('Per-class metrics file not found.')

In [ ]:
if not full_results.empty:
    plt.figure(figsize=(12,6))
    sns.boxplot(data=full_results, x='probe', y='test_macro_f1')
    plt.title('Distribution of Test Macro-F1 per Probe (all models/datasets)')
    plt.xticks(rotation=45)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# CELL 13 – EXPORT SUMMARY AND CONCLUSION
# =============================================================================
if not full_results.empty:
    summary_path = Path('probe_results_summary.csv')
    best_per_entry.to_csv(summary_path, index=False)
    print(f"Summary saved to {summary_path}")
else:
    print("No data to summarize.")

# Read The CSV file and Process the Result

In [ ]:
from pathlib import Path
import pandas as pd
from rich.console import Console
from rich.table import Table
import matplotlib.pyplot as plt
import seaborn as sns

console = Console()

# Define checkpoint directory
checkpoint_dir = Path('/Volumes/Amirali/hidden_states/experiments/baseline_v5_001/matrix_checkpoint')
per_entry_dir = checkpoint_dir / 'per_entry_results'

# Robust CSV reader
def read_csv_robust(file_path: Path) -> pd.DataFrame:
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            return pd.read_csv(file_path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(file_path, encoding='utf-8', encoding_errors='replace')

# Collect all result files
csv_files = sorted(per_entry_dir.glob('*_layer_probe_results.csv'))
frames = []
for f in csv_files:
    try:
        df = read_csv_robust(f)
    except Exception as e:
        console.print(f"[red]Failed to read {f.name}: {e}[/red]")
        continue

    # Ensure model/dataset columns
    if 'model' not in df.columns or 'dataset' not in df.columns:
        stem = f.stem.replace('_layer_probe_results', '')
        parts = stem.split('_')
        df['dataset'] = parts[-1]
        df['model'] = '_'.join(parts[:-1])

    # Ensure artifact_dir column
    if 'artifact_dir' not in df.columns:
        df['artifact_dir'] = str(f.parent)

    frames.append(df)

if frames:
    results_df = pd.concat(frames, ignore_index=True)
    console.print(f"[green]Loaded {len(frames)} result files, {len(results_df)} rows total.[/green]")
else:
    results_df = pd.DataFrame()
    console.print("[yellow]No result files found.[/yellow]")

In [ ]:
# frames

In [ ]:
# results_df

In [ ]:
if results_df.empty:
    console.print("[yellow]No data to display.[/yellow]")
else:
    single_label = results_df[results_df['task_type'] == 'single_label'].copy()
    multi_label = results_df[results_df['task_type'] == 'multi_label'].copy()

    # ---- Single-label best layers ----
    console.rule("[bold cyan]Best Layer per Probe (Single‑Label Datasets)")
    if not single_label.empty:
        best_single = single_label.loc[
            single_label.groupby(['probe', 'model', 'dataset'])['test_macro_f1'].idxmax()
        ]
        table = Table(show_header=True, header_style="bold magenta", expand=True)
        table.add_column("Model", style="cyan", no_wrap=True)
        table.add_column("Dataset", no_wrap=True)
        table.add_column("Probe", no_wrap=True)
        table.add_column("Best Layer", justify="center")
        table.add_column("Macro-F1", justify="right")
        table.add_column("Probe Score", justify="right")
        for _, row in best_single.iterrows():
            table.add_row(
                row['model'], row['dataset'], row['probe'],
                str(int(row['layer_index'])),
                f"{row['test_macro_f1']:.4f}",
                f"{row['probe_score']:.4f}"
            )
        console.print(table)
    else:
        console.print("[yellow]No single-label results found.[/yellow]")

    # ---- Multi-label best layers ----
    console.rule("[bold cyan]Best Layer per Probe (Multi‑Label Datasets)")
    if not multi_label.empty:
        best_multi = multi_label.loc[
            multi_label.groupby(['probe', 'model', 'dataset'])['test_macro_f1'].idxmax()
        ]
        table = Table(show_header=True, header_style="bold magenta", expand=True)
        table.add_column("Model", style="cyan", no_wrap=True)
        table.add_column("Dataset", no_wrap=True)
        table.add_column("Probe", no_wrap=True)
        table.add_column("Best Layer", justify="center")
        table.add_column("Macro-F1", justify="right")
        table.add_column("Exact Match Acc.", justify="right")
        for _, row in best_multi.iterrows():
            table.add_row(
                row['model'], row['dataset'], row['probe'],
                str(int(row['layer_index'])),
                f"{row['test_macro_f1']:.4f}",
                f"{row.get('test_exact_match_accuracy', float('nan')):.4f}"
            )
        console.print(table)
    else:
        console.print("[yellow]No multi-label results found.[/yellow]")

In [ ]:
def plot_layer_curves(df, task_name, metric='test_macro_f1'):
    plt.figure(figsize=(12, 6))
    for probe in sorted(df['probe'].unique()):
        sub = df[df['probe'] == probe].groupby('layer_index')[metric].mean().sort_index()
        plt.plot(sub.index, sub.values, marker='o', linewidth=2, label=probe)
    plt.xlabel('Layer Index')
    plt.ylabel(metric.replace('test_', 'Test ').replace('_', ' ').title())
    plt.title(f'Average {metric} across layers – {task_name}')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

if not single_label.empty:
    plot_layer_curves(single_label, 'Single‑Label (ISEAR)', 'test_macro_f1')
if not multi_label.empty:
    plot_layer_curves(multi_label, 'Multi‑Label (GoEmotions)', 'test_macro_f1')

In [ ]:
def plot_heatmap(df, task_name, metric='test_macro_f1'):
    pivot = df.pivot_table(index='probe', columns='layer_index', values=metric, aggfunc='mean')
    plt.figure(figsize=(14, 6))
    sns.heatmap(pivot, annot=True, fmt=".3f", cmap='viridis', cbar_kws={'label': metric})
    plt.title(f'{metric} Heatmap – {task_name}')
    plt.xlabel('Layer Index')
    plt.ylabel('Probe')
    plt.tight_layout()
    plt.show()

if not single_label.empty:
    plot_heatmap(single_label, 'Single‑Label (ISEAR)', 'test_macro_f1')
    plot_heatmap(single_label, 'Single‑Label (ISEAR)', 'probe_score')
if not multi_label.empty:
    plot_heatmap(multi_label, 'Multi‑Label (GoEmotions)', 'test_macro_f1')
    plot_heatmap(multi_label, 'Multi‑Label (GoEmotions)', 'probe_score')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
if not single_label.empty:
    sns.boxplot(data=single_label, x='probe', y='test_macro_f1', ax=axes[0])
    axes[0].set_title('Single‑Label: Macro‑F1 Distribution')
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
if not multi_label.empty:
    sns.boxplot(data=multi_label, x='probe', y='test_macro_f1', ax=axes[1])
    axes[1].set_title('Multi‑Label: Macro‑F1 Distribution')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
console.rule("[bold cyan]Average Macro‑F1 per Model and Task Type")
if not single_label.empty:
    avg_single = single_label.groupby(['model', 'dataset'])['test_macro_f1'].mean().reset_index()
    table = Table(show_header=True, header_style="bold magenta", expand=True)
    table.add_column("Model", style="cyan", no_wrap=True)
    table.add_column("Dataset", no_wrap=True)
    table.add_column("Avg Macro‑F1", justify="right")
    for _, row in avg_single.iterrows():
        table.add_row(row['model'], row['dataset'], f"{row['test_macro_f1']:.4f}")
    console.print("Single‑Label:")
    console.print(table)

if not multi_label.empty:
    avg_multi = multi_label.groupby(['model', 'dataset'])['test_macro_f1'].mean().reset_index()
    table = Table(show_header=True, header_style="bold magenta", expand=True)
    table.add_column("Model", style="cyan", no_wrap=True)
    table.add_column("Dataset", no_wrap=True)
    table.add_column("Avg Macro‑F1", justify="right")
    for _, row in avg_multi.iterrows():
        table.add_row(row['model'], row['dataset'], f"{row['test_macro_f1']:.4f}")
    console.print("Multi‑Label:")
    console.print(table)